# Multidimensional Event Selection: From Cuts to Classifiers

<CENTER><img src=\"https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/images/ATLASOD.gif?raw=1\" style=\"width:50%\"></CENTER>

*An introductory notebook on multivariate analysis techniques in particle physics, using a 2D toy example.*

This notebook focuses on a common problem in particle physics: **multidimensional event selection**:
how do we separate "signal" from "background" when no single variable does the job cleanly?

### Learning objectives

By the end of this notebook you should be able to:

1. Explain why a single one-dimensional "cut" is often not enough to separate signal from background.
2. Describe what a **binned likelihood** discriminant is, and build one from histograms.
3. Describe, at a conceptual level, how a **boosted decision tree (BDT)** combines many simple decision
   rules into a strong classifier.
4. Describe, at a conceptual level, how a **neural network** (shallow and deep) learns a decision boundary.
5. Explain why the *choice of input variables* matters at least as much as the *choice of algorithm*,
   especially when the amount of training data is limited.

We will do all of this using a simple, fully-controlled 2D toy problem, so that we always know the
exact "ground truth" answer and can judge how well each method recovers it.

## 1. Why do we need *multi*-dimensional selection?

In a typical particle physics analysis we record many properties of each collision event: the energies and
momenta of jets, leptons, missing transverse energy, angles between particles, invariant masses of
combinations of particles, and so on. Each one of these is a "variable" we could potentially use to decide
whether an event looks more like our signal process or more like one of many background processes.

The simplest possible event selection is a **single cut**: pick one variable, say the missing transverse
energy $E_T^{\rm miss}$, and require $E_T^{\rm miss} > 250$ GeV. This works well when the signal and
background distributions of that one variable are cleanly separated, for example if signal events mostly
have high $E_T^{\rm miss}$ and background mostly has low $E_T^{\rm miss}$, with little overlap.

In many realistic analyses, however, **no single variable cleanly separates signal from background.**
Instead, signal and background events occupy different, and often overlapping, *regions of a
multi-dimensional space* formed by several variables together. A region of that space might be
"somewhat signal-like" without being purely signal, or "somewhat background-like" without being purely
background. The right question to ask is not "is variable $X$ above some threshold?" but rather
"given everything we measured about this event, how signal-like is the **point** that this event
corresponds to in our multi-dimensional variable space?"

This is exactly the problem that **multivariate analysis (MVA)** techniques are designed to solve:
combine several variables together into a single "score" that tells us how signal-like or background-like
an event is, using the full multidimensional information rather than one variable at a time.

To build intuition for this, we will use a toy problem in just **two dimensions**, $x$ and $y$. Two
dimensions is small enough to visualize directly (we can easily draw a picture of the signal and
background regions and of any classifier's output), yet it is sufficient to demonstrate the concepts
that we are after.

## 2. Setup

We only need a few standard scientific python libraries: `numpy` for numerics, `matplotlib` for all of our
plots, and `scikit-learn` for the boosted decision tree and neural network models. On most platforms you can simply import these libraries; on a few platforms you might need to pip-install them first.

In [ ]:
# For numerical manipulation
import numpy as np

# For plotting
import matplotlib.pyplot as plt
import matplotlib as mpl

# For boosted decision trees and neural networks
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Make plots reasonably large and readable by default.
mpl.rcParams['figure.figsize'] = (5.5, 5)
mpl.rcParams['font.size'] = 11

## 3. Parameters

All of the "knobs" for this notebook are collected here in one place, so you can easily come back and
change them (e.g. to see what happens with a smaller or larger training sample — see the suggestions
at the end of the notebook).


In [ ]:
# --- Geometry of the problem -------------------------------------------------
X_RANGE = (-2.0, 2.0)     # range of x we consider
Y_RANGE = (-2.0, 2.0)     # range of y we consider
R_LO, R_HI = 0.95, 1.05   # the TRUE signal region is the thin ring R_LO < r < R_HI

# --- Sample sizes -------------------------------------------------------------
N_SIG_TRAIN = 10_000   # number of "signal" training points
N_BKG_TRAIN = 10_000   # number of "background" training points
N_TEST      = 100_000  # number of unlabelled evaluation/test points, drawn uniformly in the box

# --- Binning used for the binned-likelihood methods ---------------------------
N_BINS_XY   = 40   # 40 x 40 bins in (x, y)
N_BINS_RPHI = 40   # 40 x 40 bins in (r, phi)

# --- Marker size used only for DISPLAYING the evaluation sample as scatter dots ---
EVAL_POINT_SIZE = 3

# --- Reproducibility -----------------------------------------------------------
# We use many random numbers throughout this workbook. If you set this seed to a
# specific number, each run will be identical. If you set the seed to 0, the random
# number generator will be seeded based on the time of the run, so each run will be
# slightly different.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 4. The toy problem

We define our toy "signal" purely geometrically. A point $(x, y)$ is **signal** if it lies in a thin ring
(annulus) of radius between 0.95 and 1.05 around the origin:

$$
0.95 < \sqrt{x^2 + y^2} < 1.05 \qquad \Longrightarrow \qquad \text{signal}
$$

Every other point in our region of interest, $-2 \le x \le 2$ and $-2 \le y \le 2$, is **background**.

You can see immediately why simple cuts on $x$ and $y$ cannot solve this problem: a cut on $x$ alone (e.g. "$x>0.9$") keeps a
vertical strip that contains both signal and background, and likewise for a cut on $y$ alone, or for 1D
cuts on both $x$ and $y$. The signal
region is only well defined when $x$ and $y$ are considered *together*, through the combination
$r = \sqrt{x^2+y^2}$.

Let's draw the "ground truth" picture: every point in the box is colored according to whether it is truly
signal or background.

In [ ]:
def true_is_signal(x, y):
    """Ground truth definition: True for signal, False for background."""
    r = np.sqrt(x**2 + y**2)
    return (r > R_LO) & (r < R_HI)


# Evaluate the truth on a fine, regular grid just for this illustration.
grid_x = np.linspace(X_RANGE[0], X_RANGE[1], 400)
grid_y = np.linspace(Y_RANGE[0], Y_RANGE[1], 400)
grid_X, grid_Y = np.meshgrid(grid_x, grid_y)
truth_image = true_is_signal(grid_X, grid_Y).astype(float)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(truth_image, origin='lower',
               extent=[X_RANGE[0], X_RANGE[1], Y_RANGE[0], Y_RANGE[1]],
               cmap='coolwarm', vmin=0, vmax=1)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Ground truth: signal (red) vs. background (blue)')
ax.set_aspect('equal')
plt.colorbar(im, ax=ax, label='1 = signal, 0 = background')
plt.show()

## 5. Generating training and evaluation data

We need three samples:

1. **Signal training sample** ($N_{\rm sig}$ points): generated *directly* from the true signal
   definition, by drawing a radius uniformly in $[0.95, 1.05]$ and an angle uniformly in $[0, 2\pi)$,
   then converting to Cartesian coordinates. This is analogous to generating a Monte Carlo
   sample of a signal events.
2. **Background training sample** ($N_{\rm bkg}$ points): generated uniformly over the full box
   $[-2,2]\times[-2,2]$, *excluding* the signal ring (we use simple rejection sampling: draw a
   candidate point, and keep it only if it falls outside the ring).
3. **Evaluation ("test") sample** ($N_{\rm test}$ points): generated uniformly over the full box,
   **without** any labels and **without** excluding the signal ring. This represents new, unlabelled
   data — exactly the kind of sample we would apply a trained classifier to in a real analysis. We will
   use this same sample throughout the notebook to visualize and compare each method on equal footing.

All three samples are generated with the parameters defined above, so you can easily change the sample
sizes and re-run the whole notebook.

In [ ]:
def generate_signal(n, rng):
    """Draw n signal points uniformly within the ring R_LO < r < R_HI."""
    r = rng.uniform(R_LO, R_HI, n)
    phi = rng.uniform(0.0, 2 * np.pi, n)
    x = r * np.cos(phi)
    y = r * np.sin(phi)
    return x, y


def generate_background(n, rng, xrange=X_RANGE, yrange=Y_RANGE):
    """Draw n background points uniformly over the box, excluding the signal ring.

    We use rejection sampling: since the ring only covers a small fraction of the
    box's area, most random points already qualify, so this converges in a single
    extra batch or two.
    """
    xs = np.empty(0)
    ys = np.empty(0)
    while xs.size < n:
        n_needed = n - xs.size
        batch_size = max(int(n_needed * 1.2), 200)
        bx = rng.uniform(xrange[0], xrange[1], batch_size)
        by = rng.uniform(yrange[0], yrange[1], batch_size)
        r = np.sqrt(bx**2 + by**2)
        keep = (r <= R_LO) | (r >= R_HI)
        xs = np.concatenate([xs, bx[keep]])
        ys = np.concatenate([ys, by[keep]])
    return xs[:n], ys[:n]


def generate_test_points(n, rng, xrange=X_RANGE, yrange=Y_RANGE):
    """Draw n unlabelled evaluation points, uniformly over the full box."""
    x = rng.uniform(xrange[0], xrange[1], n)
    y = rng.uniform(yrange[0], yrange[1], n)
    return x, y


# Generate all three samples from the single shared random number generator.
sig_x_train, sig_y_train = generate_signal(N_SIG_TRAIN, rng)
bkg_x_train, bkg_y_train = generate_background(N_BKG_TRAIN, rng)
x_test, y_test = generate_test_points(N_TEST, rng)

print(f"Generated {len(sig_x_train):,} signal training points")
print(f"Generated {len(bkg_x_train):,} background training points")
print(f"Generated {len(x_test):,} unlabelled evaluation points")

Let's visualize the training data. Plotting signal and background training points on top of each other
should make the ring shape visible even though we have not told the plotting code anything about rings —
this is exactly the pattern that we want our classifiers to learn directly from the data.


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(bkg_x_train, bkg_y_train, s=2, c='steelblue', alpha=0.25, label='background (training)')
ax.scatter(sig_x_train, sig_y_train, s=2, c='crimson', alpha=0.25, label='signal (training)')
ax.set_xlim(X_RANGE)
ax.set_ylim(Y_RANGE)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Training data')
ax.set_aspect('equal')
ax.legend(loc='upper right', markerscale=3, framealpha=0.9)
plt.show()

## 6. A common way to visualize every method

Every method below will produce a single number — a "score" — for every point
$(x,y)$, telling us how signal-like that point looks. To compare the five methods fairly, we will always:

1. **Evaluate** the trained method's score at every one of the $N_{\rm test}$ unlabelled evaluation points.
2. **Plot the evaluation points themselves**, as a scatter plot colored by their score, with
   no further processing.

We deliberately avoid introducing an extra display-binning step (e.g. averaging the evaluation sample into
some new 50$\times$50 display grid). If we did that, *every* method's picture would inherit the same
display grid's bin edges, and it would become hard to tell whether any blockiness you see is a genuine
feature of a method (such as the native $40\times 40$ grid that actually defines the binned-likelihood
lookup tables) or merely an artifact of how we chose to draw the picture. Plotting one dot per evaluation
point side-steps this entirely: with 100,000 points spread over a modest area, the dots are dense enough to
look like a smooth, continuous heat map wherever the underlying score genuinely varies smoothly, while any
real bin edges or sharp boundaries still show up crisply, because they are truly present in the data, not
introduced by us.

In [ ]:
def plot_score_scatter(x, y, score, title, ax=None, cmap='viridis', vmin=0, vmax=1,
                        xlabel='x', ylabel='y', xrange=X_RANGE, yrange=Y_RANGE,
                        point_size=EVAL_POINT_SIZE):
    """Scatter-plot evaluation points (x, y), colored directly by their score.

    We plot the real evaluation points themselves -- not a re-binned average -- so that
    any blockiness or sharp edges visible in the plot are genuine features of the method
    being evaluated, never an artifact of our display choices.
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(5.5, 5))
    sc = ax.scatter(x, y, c=score, cmap=cmap, vmin=vmin, vmax=vmax,
                     s=point_size, edgecolors='none', rasterized=True)
    ax.set_xlim(xrange)
    ax.set_ylim(yrange)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_aspect('equal')
    plt.colorbar(sc, ax=ax, fraction=0.045, pad=0.04, label='signal-like score')
    if standalone:
        plt.show()
    return sc


def plot_grid(score_grid, aedges, bedges, title, ax=None, cmap='viridis', vmin=0, vmax=1,
              xlabel='x', ylabel='y'):
    """Plot a method's own native score grid (e.g. the bins of a binned likelihood)
    directly as an image. Unlike plot_score_scatter, this does NOT involve the
    evaluation sample at all -- it shows the actual lookup table the method is built
    from, bin by bin.
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(score_grid.T, origin='lower',
                    extent=[aedges[0], aedges[-1], bedges[0], bedges[-1]],
                    cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.045, pad=0.04, label='signal-like score')
    if standalone:
        plt.show()
    return im

## 7. Method 1 — Binned likelihood in $(x, y)$

The simplest possible multivariate method is to **histogram** the training data. We divide the
$(x,y)$ plane into a grid of $40\times 40$ bins, and in each bin we count how many signal training points
$S$ and how many background training points $B$ fall into it. A natural "signal-like score" for that bin
is then

$$
\text{score} = \frac{S}{S+B},
$$

which is close to 1 in bins dominated by signal, close to 0 in bins dominated by background, and around
0.5 in bins with a roughly equal mix (or with very little data at all). This quantity is directly related
to a likelihood-ratio discriminant: if $s(x,y)$ and $b(x,y)$ are the (binned, estimated) signal and
background probability densities, then $S/(S+B) \approx s/(s+b)$, a monotonic function of the
likelihood ratio $s/b$ commonly used to rank events from most to least signal-like.

A small smoothing term is added to both numerator and denominator to avoid division by zero in empty
bins (this is equivalent to assuming a tiny "prior" amount of signal and background in every bin).

Once we have this lookup table of one score per bin, we can evaluate it for *any* new point by simply
finding which bin that point falls into.


In [ ]:
def make_binned_likelihood(sig_a, sig_b, bkg_a, bkg_b, bins, arange, brange, smoothing=0.5):
    """Build a binned-likelihood score grid score = (S + eps) / (S + B + 2*eps)
    from signal and background training points expressed in coordinates (a, b)
    -- e.g. (a, b) = (x, y) or (a, b) = (r, phi).
    """
    sig_counts, aedges, bedges = np.histogram2d(sig_a, sig_b, bins=bins, range=[arange, brange])
    bkg_counts, _, _ = np.histogram2d(bkg_a, bkg_b, bins=bins, range=[arange, brange])
    score_grid = (sig_counts + smoothing) / (sig_counts + bkg_counts + 2 * smoothing)
    return score_grid, aedges, bedges, sig_counts, bkg_counts


def lookup_grid_score(a, b, score_grid, aedges, bedges):
    """Evaluate a binned score grid at arbitrary points (a, b) by bin lookup."""
    ai = np.clip(np.searchsorted(aedges, a, side='right') - 1, 0, score_grid.shape[0] - 1)
    bi = np.clip(np.searchsorted(bedges, b, side='right') - 1, 0, score_grid.shape[1] - 1)
    return score_grid[ai, bi]


# Train the binned likelihood directly in (x, y).
score_grid_xy, xedges_xy, yedges_xy, sigN_xy, bkgN_xy = make_binned_likelihood(
    sig_x_train, sig_y_train, bkg_x_train, bkg_y_train,
    bins=N_BINS_XY, arange=X_RANGE, brange=Y_RANGE)

# Show the RAW 40x40 grid itself: this *is* the binned-likelihood discriminant.
plot_grid(score_grid_xy, xedges_xy, yedges_xy,
          f'Binned likelihood, native {N_BINS_XY}x{N_BINS_XY} grid in (x, y)')

Now let's apply this same lookup table to every point in our 100,000-point evaluation sample, and plot
those evaluation points directly, colored by their score.


In [ ]:
score_bl_xy_test = lookup_grid_score(x_test, y_test, score_grid_xy, xedges_xy, yedges_xy)

plot_score_scatter(x_test, y_test, score_bl_xy_test, 'Binned likelihood (x, y), evaluated on test sample')

**What to notice:** the ring is clearly visible, but the map is noticeably *grainy/blocky*. Because we
are plotting the real evaluation points directly, you can see that this blockiness is made of actual square
islands of constant color, exactly the size of one $(x,y)$ bin. This is a genuine feature of the method, not
a plotting artifact: it is because the boundary of the signal ring is curved, so it cuts diagonally across
many of our $40\times 40$ square bins, and because each individual bin only contains a modest number of
training points (the ring has a circumference of about $2\pi \approx 6.3$, so it is spread out across
roughly 60–70 bins, each with only on the order of $10{,}000/65 \approx 150$ signal points, and even fewer
once you account for partial overlap with background). This statistical noise, combined with the coarse,
axis-aligned binning, is the price we pay for using a simple binned-likelihood method with limited statistics
directly in $(x,y)$ — a point we will come back to in Method 5.

This is a common issue in physics analysis: that the amount of training data is limited. Given unlimited
training data, a very finely-binned likelihood can easily reproduce an arbitrary shape, but such an
approach often requires more computing time than is available just to produce enough signal and background.

## 8. Preparing the data for machine-learning methods

The boosted decision tree and the neural networks below all expect their training data in a standard
"feature matrix + label vector" format: an array `X_train` with one row per training point and one column
per input variable, and a label array `y_train` that is 1 for signal and 0 for background. We will build
that once here, and reuse it for every ML method.

In [ ]:
X_train = np.column_stack([
    np.concatenate([sig_x_train, bkg_x_train]),
    np.concatenate([sig_y_train, bkg_y_train]),
])
y_train = np.concatenate([np.ones(N_SIG_TRAIN), np.zeros(N_BKG_TRAIN)])

X_test = np.column_stack([x_test, y_test])

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape, ' (fraction signal = %.2f)' % y_train.mean())

## 9. Method 2 — Boosted Decision Tree (BDT)

A **decision tree** classifies a point by asking a sequence of simple yes/no questions, such as
"is $x > 0.3$?" and then "is $y > -0.8$?", arranged in a tree: each question splits the data into two
groups, and each group is split further, until the tree decides a final region is "mostly signal" or
"mostly background". A single decision tree, by itself, can only carve up the plane into axis-aligned
rectangles (much like a histogram), so it is a fairly crude classifier on its own, but it is fast to
train and easy to reason about.

**Boosting** improves on this dramatically: instead of training one tree, we train *many* small trees, one
after another, where each new tree focuses on fixing the mistakes made by the trees trained before it
(formally, each new tree is fit to the residual errors of the current ensemble). The final score is a
weighted sum (here, after a logistic transformation) of the predictions of all the trees together. Boosted
trees (often called just "BDTs") are a workhorse multivariate technique in particle physics, because they
tend to work well "out of the box" with relatively little tuning, and they handle the kind of curved,
irregular decision boundary like we have in this toy problem naturally, since many small rectangles can
approximate a circle reasonably well.

Here we use `GradientBoostingClassifier` from `scikit-learn`, a standard, off-the-shelf implementation of
a gradient-boosted decision tree ensemble. You can see how straightforward it is in this case to train,
test, and plot the output thanks to these libraries.

In [ ]:
bdt = GradientBoostingClassifier(
    n_estimators=200,     # number of boosting stages (trees)
    max_depth=3,          # how many splits deep each individual tree can go
    learning_rate=0.1,    # how much each new tree is allowed to correct the ensemble
    random_state=RANDOM_SEED,
)
bdt.fit(X_train, y_train)

# Evaluate the trained BDT on every point in our unlabelled evaluation sample.
score_bdt_test = bdt.predict_proba(X_test)[:, 1]

plot_score_scatter(x_test, y_test, score_bdt_test, 'Boosted Decision Tree score, evaluated on test sample')

**What to notice:** the BDT recovers a much smoother ring than the plain binned likelihood, with far less
graininess. This is because the BDT does not commit to one fixed grid of bins ahead of time; it can place
its splits anywhere, and because boosting lets it combine many small, simple regions into a much more
refined approximation of the true (curved) boundary. The price is that a BDT is a more "black box" model
than a histogram: we can no longer simply read off $S$ and $B$ in a bin, although tools exist to inspect
feature importance and partial decision paths if needed.

## 10. Method 3 — A shallow neural network

A (feed-forward) **neural network** builds a classifier out of layers of simple units ("neurons"). Each
neuron computes a weighted sum of its inputs, adds a bias, and passes the result through a nonlinear
"activation function" (here, the common choice ReLU, $f(z)=\max(0,z)$). A network with a single hidden
layer of a modest number of neurons is often called a **shallow** network. Even with just one hidden
layer, a neural network with enough neurons and infinite training data can in principle approximate any
reasonably smooth decision boundary — this is sometimes called the "universal approximation" property —
but a shallow network with few neurons has a limited "vocabulary" of shapes it can build.

The network's weights and biases are learned by **training**: we repeatedly show it labelled examples, see
how wrong its predictions are (the "loss"), and nudge every weight slightly in the direction that reduces
that error (using the *backpropagation* algorithm to compute how much each weight contributed to the
error, combined with gradient descent to update the weights).

Here we use `MLPClassifier` ("multi-layer perceptron") from `scikit-learn` with a single hidden layer of
just 8 neurons — intentionally small, so we can later compare it to a *deep* network.

Neural networks generally train better when input features are rescaled to a similar numerical range, so
we first standardize $x$ and $y$ to zero mean and unit variance using `StandardScaler` (fit only on the
training data, then applied identically to the evaluation sample).

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

nn_shallow = MLPClassifier(
    hidden_layer_sizes=(8,),   # ONE hidden layer with 8 neurons -- intentionally small/"shallow"
    activation='relu',
    max_iter=2000,
    random_state=RANDOM_SEED,
)
nn_shallow.fit(X_train_scaled, y_train)

score_nn_shallow_test = nn_shallow.predict_proba(X_test_scaled)[:, 1]

plot_score_scatter(x_test, y_test, score_nn_shallow_test,
                    'Shallow neural network score (1 hidden layer, 8 neurons)')

**What to notice:** with only 8 hidden neurons, the network simply does not have enough representational
capacity to trace out a thin ring. Each hidden neuron in a single-layer network effectively contributes
one straight-line cut, and combining only a handful of these naturally produces a single polygon-shaped
region rather than two independent boundaries (an inner edge *and* an outer edge). Faced with this
limitation, the network has settled for marking almost the *entire interior disk* enclosed by the ring as
moderately "signal-like" (notice the score plateaus around 0.7–0.8, not 1.0) rather than tracing the thin
annulus itself — a reasonable compromise given its limited flexibility, but a poor approximation to the
truth. This shows up clearly in the quantitative comparison in Section 13, where the shallow network scores
noticeably worse than every other method. Try increasing the number of neurons in the hidden layer (or
adding a second hidden layer) and see how the shape improves!

## 11. Method 4 — A deep neural network

A **deep** neural network simply has *more* hidden layers stacked on top of each other. Each additional
layer lets the network build increasingly complex features out of the features learned by the previous
layer. For example, an early layer might learn simple linear boundaries, and a later layer might learn to
combine several of those into smoothly curved shapes. In practice, depth (combined with enough neurons per
layer and enough training data) tends to give a network the capacity to represent much more intricate
decision boundaries than a shallow network of similar total size.

Here we use the same `MLPClassifier` architecture as before, but with **four** hidden layers of 64 neurons
each, instead of one hidden layer of 8 neurons — a substantially larger and deeper model.

In [ ]:
nn_deep = MLPClassifier(
    hidden_layer_sizes=(64, 64, 64, 64),   # FOUR hidden layers -- a "deep" network
    activation='relu',
    max_iter=2000,
    random_state=RANDOM_SEED,
)
nn_deep.fit(X_train_scaled, y_train)

score_nn_deep_test = nn_deep.predict_proba(X_test_scaled)[:, 1]

plot_score_scatter(x_test, y_test, score_nn_deep_test,
                    'Deep neural network score (4 hidden layers, 64 neurons each)')


**What to notice:** the deep network typically produces a noticeably smoother, more accurate ring than the
shallow network, since it has far more capacity to represent a curved boundary. With *unlimited* training
data, both networks (and the BDT, and even the binned likelihood) should eventually converge to the same
underlying truth, since they are all, in the end, just trying to approximate the same function
$P(\text{signal} \mid x, y)$. With a *finite* amount of training data (here, only 10,000 signal and 10,000
background points), however, more flexible models can sometimes only be trained reliably if there is enough
data to constrain all of their extra parameters. Otherwise they risk **overfitting**: fitting noise in the
training sample rather than the true underlying pattern. Try re-running this notebook with a much smaller
`N_SIG_TRAIN`/`N_BKG_TRAIN` (see the suggestions at the end) or a much larger network  and see how the deep
network's heat map changes!

## 12. Method 5 — Choosing better input variables: binned likelihood in $(r, \varphi)$

So far we have only changed the *algorithm* (histogram, BDT, shallow network, deep network) while always
feeding it the raw coordinates $(x, y)$. But there is another, often more powerful, lever available to us:
**changing the input variables themselves**.

With infinite training data, every method above should eventually converge to the same, correct answer,
because they are all general-purpose function approximators that can, given enough examples, learn the
exact shape of the true signal region regardless of which coordinates we hand them. But real analyses never
have infinite training data, and different methods make different trade-offs in how *efficiently* they use
the data they have. A method that has to spend its limited statistics on learning a complicated,
arbitrarily-curved boundary (like the ring boundary expressed in $x$ and $y$) will generally do worse, for
a given amount of training data, than a method that is given variables in which the *true* boundary is
simple.

Here, this is easy to see: our signal region $0.95 < r < 1.05$ is a simple **band** in the variable
$r = \sqrt{x^2+y^2}$, with **no dependence at all** on the angle $\varphi = \arctan(y/x)$. If we re-express
our training data in polar coordinates $(r, \varphi)$ and build a $40\times 40$ binned likelihood exactly as
we did in Method 1 — but now in $(r,\varphi)$ instead of $(x,y)$ — the signal events, instead of being
smeared thinly around the circumference of a circle across ~60 different bins, pile up into just a couple of
*rows* of the histogram (a narrow band in $r$), each of which spans *all* 40 columns in $\varphi$ and
therefore contains many more signal events per occupied bin. The decision boundary itself becomes a simple,
axis-aligned threshold in $r$ — exactly the kind of boundary that even the crudest binned method can
capture with very little statistical noise.

Physically, this corresponds to recognizing that our problem has a built-in symmetry — rotational
invariance around the origin — and choosing variables that respect that symmetry.

In [ ]:
def to_polar(x, y):
    r = np.sqrt(x**2 + y**2)
    phi = np.arctan2(y, x)
    return r, phi


# The corner of our box is the farthest possible point from the origin.
R_RANGE = (0.0, np.sqrt(X_RANGE[1]**2 + Y_RANGE[1]**2))
PHI_RANGE = (-np.pi, np.pi)

sig_r_train, sig_phi_train = to_polar(sig_x_train, sig_y_train)
bkg_r_train, bkg_phi_train = to_polar(bkg_x_train, bkg_y_train)

score_grid_rphi, redges, phiedges, sigN_rphi, bkgN_rphi = make_binned_likelihood(
    sig_r_train, sig_phi_train, bkg_r_train, bkg_phi_train,
    bins=N_BINS_RPHI, arange=R_RANGE, brange=PHI_RANGE)

# Show the RAW 40x40 grid -- but now its native space is (r, phi), not (x, y)!
plot_grid(score_grid_rphi, redges, phiedges,
          f'Binned likelihood, native {N_BINS_RPHI}x{N_BINS_RPHI} grid in (r, phi)',
          xlabel=r'r', ylabel=r'$\varphi$')


Notice how clean this native $(r,\varphi)$ histogram already looks: a sharp, almost perfectly uniform band
in $\varphi$ at the true radius, with very little of the graininess we saw in Method 1.

Two subtleties are worth pointing out, since they are genuinely instructive:

* You may notice faint, blotchy patches with an intermediate (grey-teal) score at large $r$, but only for
  *some* values of $\varphi$. These are $(r,\varphi)$ combinations that are geometrically impossible inside
  our square box — for example, $r=2.5$ in the direction $\varphi=0$ would correspond to the point
  $(2.5, 0)$, which lies outside $-2\le x\le 2$. Such bins receive *no* training statistics at all (neither
  signal nor background), so our smoothing formula falls back to an uninformative default near 0.5. Since
  no real $(x,y)$ point can ever fall there either, this has no effect on anything downstream.
* The histogram looks slightly noisier near $r=0$ (the left edge) than near $r=1$. This is a real, physical
  effect: even though our background was generated uniformly in *area* (i.e. uniformly in $x$ and $y$),
  the area of a thin ring of radius $r$ and width $dr$ is $dA = 2\pi r\, dr$, which *shrinks* as
  $r \to 0$. So a uniform-in-area sample naturally contains fewer points per unit $r$ near the center than
  at larger radius — a Jacobian effect of the coordinate transformation itself, not a flaw in the method.
  This is a good reminder that changing variables can reshape your statistics, not just relabel your axes.

Now let's transform our evaluation sample to $(r,\varphi)$, look up each point's score in this grid, and
plot those same evaluation points back in the familiar $(x,y)$ plane, colored by score, for a direct,
apples-to-apples comparison with the other four methods.


In [ ]:
r_test, phi_test = to_polar(x_test, y_test)
score_bl_rphi_test = lookup_grid_score(r_test, phi_test, score_grid_rphi, redges, phiedges)

plot_score_scatter(x_test, y_test, score_bl_rphi_test,
                    'Binned likelihood (r, phi), evaluated on test sample, shown in (x, y)')


**What to notice:** the ring band itself is just as crisp as the best of the other four methods — produced
by the *simplest possible* algorithm (a plain histogram ratio, with no boosting and no neural network
training at all). You can also see the noisy region near $x=y=0$ we predicted above, now visible as
individual wedge-shaped patches radiating out from the center — each wedge is literally one bin in
$\varphi$, made visible because the few low-statistics, randomly-fluctuating bins near $r=0$ happen to sit
right next to each other in angle. This is a good example of how plotting the real evaluation points,
rather than a smoothed display grid, lets us see the method's own bin structure directly.

The lesson is not that binned likelihoods are secretly the best general-purpose method; it is that **no
algorithm can fully make up for a poor choice of input variables when training data is limited**. Conversely,
choosing variables that are well matched to the physics of the problem (here, exploiting the rotational
symmetry of the signal definition) can make even a very simple method work extremely well — and, as we will
see next, the cleanest comparison is between the *same* binned-likelihood algorithm run on two different
choices of variables.


## 13. A quantitative comparison

The heat maps above let us *visually* compare the five methods, but we can also compare them quantitatively.
Because this is a toy problem, we happen to know the exact ground truth for every evaluation point (in a
real analysis we would generally not know this for real data!). This lets us compute, purely for the sake
of this exercise, the true signal/background label for every evaluation point and use it to calculate the
**ROC AUC** (area under the receiver-operating-characteristic curve) for each method's score, a standard
single-number summary of classification performance, where 1.0 is a perfect classifier and 0.5 is no better
than random guessing.

In [ ]:
true_label_test = true_is_signal(x_test, y_test).astype(int)

method_scores = {
    'Binned likelihood (x, y)':   score_bl_xy_test,
    'Boosted decision tree':      score_bdt_test,
    'Shallow neural network':     score_nn_shallow_test,
    'Deep neural network':        score_nn_deep_test,
    'Binned likelihood (r, phi)': score_bl_rphi_test,
}

print(f"{'Method':30s}  ROC AUC")
print('-' * 45)
for name, score in method_scores.items():
    auc = roc_auc_score(true_label_test, score)
    print(f"{name:30s}  {auc:.4f}")


With this much training data, most methods already perform quite well, and the deep neural network (by
far the most flexible model we tried) typically comes out on top, while the shallow network (limited to a
single polygon-like region, as we saw above) lags clearly behind everything else.

The most instructive comparison, though, is between the two *binned likelihood* entries: they use the
**exact same algorithm**, with the *only* difference being the choice of input variables. You should find
that the $(r,\varphi)$ version comes out ahead of the $(x,y)$ version (often even edging out
the boosted decision tree), despite being by far the simplest of the five methods. 

That is one of the key lessons: holding the algorithm fixed and changing only the input variables already
measurably changes performance. This is the same feature found in many introductory physics homework problems,
in which identifying the appropriate coordinates for the problem make it straightforward. In data analysis,
this corresponds to applying physical intuition to identify well-known complex variables at the outset, rather
than simply putting all the variables one can think of into a deep network and hoping for the best. In many
notebook analyses built on the ATLAS Open Data such variables can be found: invariant masses built from several
particles can in principle be learned based on the 4-vectors of the particles themselves, but providing the
invariant mass directly to the discriminant is far more powerful. The advantage becomes much more dramatic with 
less training data — try setting `N_SIG_TRAIN = N_BKG_TRAIN = 100` near the top of the notebook and re-running:
you should see the $(x,y)$ binned likelihood's AUC degrade sharply, while the $(r,\varphi)$ version holds up
much better, because each of its bins still contains reasonable statistics.

## 14. Side-by-side summary

Finally, let's put the ground truth and all five methods' heat maps side by side, for a direct visual
comparison.


In [ ]:
panels = [
    ('Ground truth',               true_label_test.astype(float)),
    ('Binned likelihood (x, y)',   score_bl_xy_test),
    ('Boosted decision tree',      score_bdt_test),
    ('Shallow neural network',     score_nn_shallow_test),
    ('Deep neural network',        score_nn_deep_test),
    ('Binned likelihood (r, phi)', score_bl_rphi_test),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, (title, score) in enumerate(panels):
    plot_score_scatter(x_test, y_test, score, title, ax=axes[i])
    # Overlay the true ring boundary as a dashed guide.
    circ_lo = plt.Circle((0, 0), R_LO, fill=False, ls='--', color='white', lw=1.2)
    circ_hi = plt.Circle((0, 0), R_HI, fill=False, ls='--', color='white', lw=1.2)
    axes[i].add_patch(circ_lo)
    axes[i].add_patch(circ_hi)

fig.tight_layout()
plt.show()


## 15. Conclusions

* Signal and background often cannot be separated by a single one-dimensional cut; instead, we need to
  reason about *regions* of a multidimensional variable space that are more or less signal-like.
* A **binned likelihood** (multi-dimensional histogram ratio) is conceptually the simplest multivariate
  method, and is easy to interpret bin-by-bin, but suffers from limited resolution and statistical noise
  when the boundary between signal and background is complicated relative to the chosen variables.
* **Boosted decision trees** and **neural networks** are more flexible, data-driven methods that can learn
  curved decision boundaries directly from training data, generally with much less graininess than a simple
  histogram for a fixed amount of training data.
* **Deeper** neural networks have more representational capacity than **shallow** ones, and can capture more
  intricate boundaries, but more flexible models generally need correspondingly more training data to be
  constrained well, and are more prone to overfitting when data is scarce.
* With *infinite* training data, all of these methods should, in principle, converge to the exact same
  optimal classifier, but with realistic, *finite* training samples, the choice of method (and especially
  the choice of **input variables**) has a large practical impact on performance.
* The single biggest lever available to us, even before choosing an algorithm, is **choosing input
  variables that are well matched to the physics of the problem.** Here, recognizing the built-in rotational
  symmetry of our toy signal and switching to polar coordinates let even the simplest possible method
  (a binned likelihood) outperform much more sophisticated algorithms run on the "wrong" variables.

### Questions to think about

1. Why does the binned likelihood in $(x,y)$ look "blocky" near the edges of the ring, while the boosted
   decision tree and neural networks look smoother? What roles do *binning* and *statistics per bin* play?
2. The shallow network with only 8 neurons ended up marking almost the entire disk *inside* the ring as
   "signal-like," rather than tracing the thin ring itself. Why might a network with very limited capacity
   prefer this kind of compromise? What happens if you increase the number of neurons in the hidden layer,
   or add a second hidden layer?
3. We trained the deep network with four hidden layers of 64 neurons, which is quite a large model for a 2D
   problem. What do you expect to happen to its heat map if you reduce `N_SIG_TRAIN` and `N_BKG_TRAIN` to,
   say, 200 each? Try it! What if you increase the number of neurons and layers to, for example, 10 layers
   with 500 neurons each? Is it the same?
4. In Method 5, the signal region was *exactly* a band in $r$, with no dependence on $\varphi$ at all,
   which is why polar coordinates worked so well. Can you think of a different (still rotationally
   symmetric) signal/background definition where polar coordinates would *not* help, or where some other
   choice of variables would be the right one?
5. We measured performance using ROC AUC, computed against a ground truth we happen to know because this
   is a toy problem. In a real analysis, how might you estimate a classifier's performance on real data,
   where you generally don't know the true label of every event?
6. We mentioned on example of a case where this comes up in physics analyses: an invariant mass. Another
   example is opening angles between two objects, where the difference in $\phi$ is more important than
   $\phi$ itself. Can you think of a reason why these opening angles might be particularly tricky to
   learn? (Hint: if one object has $\phi=6.2$ and a second has $\phi=0.1$, how far apart are they?) Can
   you think of other variables used in physics analysis that would be similarly difficult to learn
   directly from simple training data, but which could instead be fed directly into discriminants?

### Ideas for further exploration

* **Change the sample sizes.** Reduce `N_SIG_TRAIN`/`N_BKG_TRAIN` by 10x or 100x and re-run the notebook.
  Which methods degrade most gracefully, and which degrade fastest?
* **Change the geometry of the problem.** Try a signal region that is, for example, two separate rings at
  different radii, a thin annulus that is only present for $0 < \varphi < \pi$ (i.e. not rotationally
  symmetric), a spiral, or a signal density that varies smoothly (rather than being a sharp on/off region).
  For an asymmetric or non-circular shape, does switching to polar coordinates still help as much?
* **Add irrelevant variables.** Extend the toy problem to three or more dimensions by adding one or more
  extra coordinates that carry no information at all about signal vs. background, and see how each method's
  performance is affected — this is closely related to the "curse of dimensionality."
* **Add redundant variables.** Try adding $x^2$ as an input variable, for example. There is no additional
  information, so the performance should be the same. Is it? What if you add other arbitrary combinations
  of the input variables (like $x+y$)?
* **Try different architectures/hyperparameters.** Vary the number of trees and tree depth for the BDT, or
  the number of layers and neurons per layer for the neural networks, and see how the heat maps and AUC
  values change.
* **Try varying the random number seeds.** Do you see different performance in the ROC curves? When doing
  a physics analysis, what do you think would be an appropriate thing to do, given this randomness?
* **Try other libraries.** This notebook used `scikit-learn` throughout for simplicity, but you could try
  `xgboost` or `lightgbm` for boosted trees, or `keras`/`tensorflow` or `pytorch` for the neural networks,
  and compare.
* **Look at over-training directly.** Compare a method's performance (e.g. ROC AUC) on its own *training*
  sample versus on the independent *evaluation* sample, for models of increasing flexibility. This is a
  standard, simple way to check for overfitting.
  
<div class=\"alert alert-block alert-info\">
We welcome your feedback on this notebook or any of our other materials! Please <a href=\"https://forms.gle/zKBqS1opAHHemv9U7\">fill out this survey</a> to let us know how we're doing, and you can enter a raffle to win some <a href=\"https://atlas-secretariat.web.cern.ch/merchandise\">ATLAS merchandise</a>!
</div>